# Manual Retrieval-Augmented Generation

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline
from scratch without LangChain or a vector database.

The objective is to understand the three main stages of RAG:

1. Indexing
2. Retrieval
3. Generation

In [1]:
!pip install -q sentence-transformers google-genai

In [2]:
import numpy as np

from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import userdata

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gemini-3.5-flash-lite"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

## Indexing

### Knowledge base

In [4]:
document = """
Amazon S3 is an object storage service designed to store and retrieve large amounts of data.
Data in S3 is stored as objects inside buckets.

Amazon EC2 provides resizable virtual computing capacity in the cloud.
Users can launch virtual machines called instances.

AWS Lambda is a serverless compute service that runs code in response to events.
It does not require users to provision or manage servers.

Amazon RDS is a managed relational database service.
It supports relational database engines such as PostgreSQL, MySQL and MariaDB.

Amazon CloudFront is a content delivery network.
It distributes content through edge locations to reduce latency.
"""

### Chunking

In [5]:
chunks = [
    chunk.strip()
    for chunk in document.split("\n\n")
    if chunk.strip()
]

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:")
    print(chunk)
    print()

Chunk 0:
Amazon S3 is an object storage service designed to store and retrieve large amounts of data.
Data in S3 is stored as objects inside buckets.

Chunk 1:
Amazon EC2 provides resizable virtual computing capacity in the cloud.
Users can launch virtual machines called instances.

Chunk 2:
AWS Lambda is a serverless compute service that runs code in response to events.
It does not require users to provision or manage servers.

Chunk 3:
Amazon RDS is a managed relational database service.
It supports relational database engines such as PostgreSQL, MySQL and MariaDB.

Chunk 4:
Amazon CloudFront is a content delivery network.
It distributes content through edge locations to reduce latency.



### Embeddings

In [6]:
chunk_embeddings = embedding_model.encode(
    chunks,
    normalize_embeddings=True
)

print(chunk_embeddings.shape)

(5, 384)


## Retrieval

### Query embedding

In [7]:
question = "Which AWS service should I use to store files?"

In [8]:
query_embedding = embedding_model.encode(
    question,
    normalize_embeddings=True
)

### Similarity scores and top-k

In [9]:
scores = np.dot(
    chunk_embeddings,
    query_embedding
)

print(scores)

[0.568931   0.32512936 0.42679274 0.4790825  0.40776628]


In [10]:
top_k = 2

top_indices = np.argsort(scores)[::-1][:top_k]

for index in top_indices:
    print(f"Score: {scores[index]:.4f}")
    print(chunks[index])
    print()

Score: 0.5689
Amazon S3 is an object storage service designed to store and retrieve large amounts of data.
Data in S3 is stored as objects inside buckets.

Score: 0.4791
Amazon RDS is a managed relational database service.
It supports relational database engines such as PostgreSQL, MySQL and MariaDB.



### Context

In [11]:
retrieved_chunks = [
    chunks[index]
    for index in top_indices
]

context = "\n\n".join(retrieved_chunks)

print(context)

Amazon S3 is an object storage service designed to store and retrieve large amounts of data.
Data in S3 is stored as objects inside buckets.

Amazon RDS is a managed relational database service.
It supports relational database engines such as PostgreSQL, MySQL and MariaDB.


## Generation

### RAG prompt

In [12]:
prompt = f"""
Answer the question using only the context provided below.

If the answer cannot be found in the context, say:
"The information is not available in the provided context."

Context:
{context}

Question:
{question}
"""

In [13]:
print(prompt)


Answer the question using only the context provided below.

If the answer cannot be found in the context, say:
"The information is not available in the provided context."

Context:
Amazon S3 is an object storage service designed to store and retrieve large amounts of data.
Data in S3 is stored as objects inside buckets.

Amazon RDS is a managed relational database service.
It supports relational database engines such as PostgreSQL, MySQL and MariaDB.

Question:
Which AWS service should I use to store files?



### LLM call

In [14]:
api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

In [18]:
response = client.models.generate_content(
    model=LLM_MODEL,
    contents=prompt
)

print(response.text)

Based on the provided context, you should use Amazon S3, which is an object storage service designed to store and retrieve large amounts of data (stored as objects inside buckets).


## Grounding Test

The following experiment checks how the RAG pipeline behaves when the knowledge base does not contain the requested information.

In [19]:
question_no_context = "Which AWS service should I use to run Kubernetes clusters?"

In [20]:
query_embedding_no_context = embedding_model.encode(
    question_no_context,
    normalize_embeddings=True
)

In [21]:
scores_no_context = np.dot(
    chunk_embeddings,
    query_embedding_no_context
)

In [22]:
top_k = 2

top_indices_no_context = np.argsort(scores_no_context)[::-1][:top_k]

for index in top_indices_no_context:
    print(f"Score: {scores_no_context[index]:.4f}")
    print(chunks[index])
    print()

Score: 0.4691
AWS Lambda is a serverless compute service that runs code in response to events.
It does not require users to provision or manage servers.

Score: 0.4430
Amazon EC2 provides resizable virtual computing capacity in the cloud.
Users can launch virtual machines called instances.



In [23]:
retrieved_chunks_no_context = [
    chunks[index]
    for index in top_indices_no_context
]

context_no = "\n\n".join(retrieved_chunks_no_context)

print(context_no)

AWS Lambda is a serverless compute service that runs code in response to events.
It does not require users to provision or manage servers.

Amazon EC2 provides resizable virtual computing capacity in the cloud.
Users can launch virtual machines called instances.


In [24]:
prompt_no_context = f"""
Answer the question using only the context provided below.

If the answer cannot be found in the context, say:
"The information is not available in the provided context."

Context:
{context_no}

Question:
{question_no_context}
"""

In [25]:
response_no_context = client.models.generate_content(
    model=LLM_MODEL,
    contents=prompt_no_context
)

print(response_no_context.text)

The information is not available in the provided context.


## RAG vs Non-RAG Comparison

In [26]:
response_without_rag = client.models.generate_content(
    model=LLM_MODEL,
    contents=(
    "Which AWS service should I use to run Kubernetes clusters? "
    "Answer in one sentence."
)

)

print(response_without_rag.text)

You should use **Amazon Elastic Kubernetes Service (Amazon EKS)** to run Kubernetes clusters on AWS.


## Key Takeaways

- RAG separates retrieval from generation.
- Documents are split into chunks and represented as embeddings during indexing.
- A query embedding is compared against chunk embeddings to retrieve relevant context.
- Retrieved context is added to the LLM prompt before generation.
- Grounding can restrict answers to information available in the retrieved context.
- Without grounding, the LLM can rely on its own parametric knowledge.